# No Peeking: reproduce on Kaggle (2x T4, free tier)

Settings: *Accelerator = GPU T4 x2*, *Internet = on*. Add a Kaggle secret `SEC_USER_AGENT` = `Your Name your@email` (SEC fair-access policy).
Each step caches its outputs, so the notebook can be re-run after a session timeout.

In [ ]:
!git clone -q https://github.com/KvnFltr/no-peeking.git
%cd no-peeking
!pip -q install -e ".[train]" bitsandbytes
import os
from kaggle_secrets import UserSecretsClient
os.environ['NOPEEKING_SEC_USER_AGENT'] = UserSecretsClient().get_secret('SEC_USER_AGENT')

## 1. Dataset (pilot: ~500 filings) and baselines

In [ ]:
!nopeeking -v data build --config configs/pilot.yaml
!nopeeking baselines --config configs/pilot.yaml

## 2. Look-ahead probe (RQ1)
Zero-shot students and teacher, original vs anonymized, pre vs post cutoff.

In [ ]:
!nopeeking probe Qwen/Qwen2.5-0.5B-Instruct --config configs/pilot.yaml --batch-size 8 --max-input-tokens 1024
!nopeeking probe Qwen/Qwen2.5-1.5B-Instruct --config configs/pilot.yaml --batch-size 8 --max-input-tokens 1024
!nopeeking probe Qwen/Qwen2.5-7B-Instruct --config configs/pilot.yaml --dtype 4bit --batch-size 4 --max-input-tokens 1024

## 3. Hindsight rationales (RQ2) with the 7B teacher in 4-bit

In [ ]:
!nopeeking -v rationales --config configs/train/rationales.yaml
!cat data/processed/pilot/rationales/*_summary.json

## 4. Post-training (LoRA) and evaluation on val + post-cutoff test
Two GPUs: prefix with `accelerate launch --num_processes 2 -m nopeeking.cli` for data-parallel runs.

In [ ]:
for run in ['sft_direct', 'sft_control', 'sft_hindsight', 'dpo', 'grpo_outcome', 'grpo_grounded', 'grpo_full']:
    !nopeeking -v train {run.split('_')[0]} --config configs/train/{run}.yaml
    !nopeeking eval-adapter --config configs/train/{run}.yaml

## 5. Report (tables, hero figure, LaTeX macros, README block)

In [ ]:
!nopeeking report --config configs/pilot.yaml
from IPython.display import Image, Markdown
display(Image('results/pilot/figures/hero.png'))
display(Markdown(open('results/pilot/summary.md').read()))